# W6 Homework — A New Intent, End to End

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week06/W6_hw_new_intent.ipynb)

**Goal.** Push one new customer intent — an exchange — through a condensed version
of the lab's pipeline: a tool for it, a planner that knows when to plan it, and a
scored request that proves it works — extending a working pipeline without breaking
its score.

The path: setup → the condensed store and pipeline → three scored intents
(baseline) → the exchange tool ✍️ → the planner extension ✍️ → four intents scored →
completion.

*Runtime:* ~45 minutes. Due before the W7 session. Reference answers:
`labs/checkpoints/week06/solution.py`, published after the homework deadline.


## 1. Setup

*Do:* run the three cells; the last must print `ready`.


In [ ]:
%pip install -q "aisuite[openai,anthropic]"


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # alt: "anthropic:claude-haiku-4-5"


In [ ]:
import aisuite

client = aisuite.Client()


def ask(prompt, system=None, temperature=0.0, **kwargs):
    """Single prompt -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=MODEL, messages=messages,
                                              temperature=temperature, **kwargs)
    return response.choices[0].message.content


print(ask("Reply with exactly: ready"))


## 2. The Store and the Pipeline, Condensed

The lab's shape in miniature: a product table, a ledger, three tools, a planner
that emits a strict-JSON plan of tool calls, and an executor that is plain code —
it runs plans and never invents them. (The reflection and explanation stages of the
lab are dropped here; the handoff discipline is the same.)

*Do:* run both cells, then read `PLANNER_SYSTEM` once — note that it names every
tool and when to use it. That list is the planner's whole world.


In [ ]:
import copy
import json

STORE0 = {
    "blue mug": {"stock": 10, "price": 9.0},
    "red mug":  {"stock": 4,  "price": 9.0},
    "tea pot":  {"stock": 2,  "price": 24.0},
}


def make_tools(store, ledger):
    """Builds the tool registry over one store instance."""

    def check_stock(product: str) -> str:
        """Report the stock level and unit price for one product.

        Use for availability or price questions; changes nothing.

        Args:
            product: exact product name in lowercase, e.g. "blue mug".
        """
        item = store.get(product)
        return (f"{product}: {item['stock']} in stock at {item['price']}"
                if item else f"(unknown product {product!r})")

    def sell_item(product: str, qty: int) -> str:
        item = store.get(product)
        if item is None:
            return f"(unknown product {product!r})"
        if item["stock"] < int(qty):
            return f"(cannot sell {qty}: only {item['stock']} in stock)"
        item["stock"] -= int(qty)
        ledger.append({"action": "sale", "product": product, "qty": int(qty)})
        return f"sold {qty} x {product}"

    def return_item(product: str, qty: int) -> str:
        item = store.get(product)
        if item is None:
            return f"(unknown product {product!r})"
        item["stock"] += int(qty)
        ledger.append({"action": "return", "product": product, "qty": int(qty)})
        return f"returned {qty} x {product}"

    return {"check_stock": check_stock, "sell_item": sell_item,
            "return_item": return_item}


In [ ]:
PLANNER_SYSTEM = """You plan tool calls for a store assistant. Reply with STRICT JSON only:
a list of steps, each {"tool": "<name>", "args": {...}}. No prose, no code fences.

Tools:
- check_stock(product): read a product's stock and price. Use for inquiries.
- sell_item(product, qty): sell qty units. Use for purchases.
- return_item(product, qty): take qty units back. Use for returns.

Products: "blue mug", "red mug", "tea pot". Use these exact names."""


def plan(request, planner_system):
    """Request -> list of {"tool", "args"} steps (raises on non-JSON)."""
    reply = ask(f"Request: {request}", system=planner_system)
    return json.loads(reply)


def execute(steps, tools):
    """Runs a plan through the registry; unknown tools become error strings."""
    results = []
    for step in steps:
        fn = tools.get(step.get("tool"))
        results.append(fn(**step.get("args", {})) if fn
                       else f"(unknown tool {step.get('tool')!r})")
    return results


## 3. Three Intents, Scored — the Baseline

Each request runs on a fresh store, and the scorer checks outcomes
programmatically: stock deltas and ledger rows, never prose.

*Do:* run the cell; the baseline must be 3/3 before you extend anything —
the same regression rule as the lab. If a request fails, re-run the cell once
(plans are sampled); if it keeps failing, print the plan the planner emitted for
that request and read it against the Tools list.


In [ ]:
REQUESTS = [
    {"request": "Sell two blue mugs to the customer at the counter.",
     "expect": lambda s, l: s["blue mug"]["stock"] == 8
               and any(r["action"] == "sale" and r["product"] == "blue mug" for r in l)},
    {"request": "How many tea pots do we have in stock?",
     "expect": lambda s, l: s == STORE0 and not l},
    {"request": "A customer returns one red mug.",
     "expect": lambda s, l: s["red mug"]["stock"] == 5
               and any(r["action"] == "return" for r in l)},
]


def score(requests, planner_system, extra_tools=None):
    correct = 0
    for case in requests:
        store, ledger = copy.deepcopy(STORE0), []
        tools = make_tools(store, ledger)
        if extra_tools:
            tools.update(extra_tools(store, ledger))
        try:
            results = execute(plan(case["request"], planner_system), tools)
            ok = case["expect"](store, ledger)
        except Exception as exc:
            results, ok = [f"(pipeline error: {exc})"], False
        correct += ok
        print(f"{'PASS' if ok else 'FAIL':4}  {case['request'][:58]}")
    return correct


baseline = score(REQUESTS, PLANNER_SYSTEM)
print(f"\nbaseline: {baseline}/3")


## 4. The Exchange Tool ✍️

An exchange is a return and a sale that must succeed or fail together: if the new
item is out of stock, the old one must not be taken back. The body below implements
that; the graded work is the docstring and the planner line you distill from it in
Section 5 — there, the Tools list is the only text the planner reads (in W3's
tool-calling setup, the docstring itself is what the model reads). Copy the shape of
`check_stock`'s docstring in Section 2.

Requirements: say what the tool does and when to use it (a customer swapping one
product for another); say when **not** to use it (plain returns and plain
purchases have their own tools); an `Args:` entry per parameter with the exact
product-name format.


In [ ]:
def make_exchange_tool(store, ledger):

    ### FILL IN (START) ###
    def exchange_item(old_product: str, new_product: str, qty: int) -> str:
        """Exchange."""
        new = store.get(new_product)
        if new is None or new["stock"] < int(qty):
            return f"(cannot exchange: {new_product!r} unavailable)"
        store[old_product]["stock"] += int(qty)
        ledger.append({"action": "return", "product": old_product, "qty": int(qty)})
        new["stock"] -= int(qty)
        ledger.append({"action": "sale", "product": new_product, "qty": int(qty)})
        return f"exchanged {qty} x {old_product} for {new_product}"
    ### FILL IN (END) ###

    return {"exchange_item": exchange_item}


## 5. The Planner Extension ✍️

The planner cannot plan a tool it has never heard of. Append the exchange tool to
`PLANNER_SYSTEM`: one line in the Tools list, in the same format as the other
three — name, signature, one sentence of when to use it. Write the line without a
leading dash; the next cell prepends `- ` itself.

Hint: the planner's failure mode without this line is instructive — run Section 6
once with the starter and read what plan it emits for the exchange request.


In [ ]:
### FILL IN (START) ###
EXCHANGE_LINE = ""   # starter — one Tools-list line for exchange_item
### FILL IN (END) ###

PLANNER_SYSTEM_V2 = PLANNER_SYSTEM.replace(
    "\nProducts:", f"- {EXCHANGE_LINE}\n\nProducts:") if EXCHANGE_LINE else PLANNER_SYSTEM
print(PLANNER_SYSTEM_V2)


## 6. Four Intents, Scored

The exchange request joins the set. Its expectation: blue-mug stock up one,
tea-pot stock down one, and both ledger rows present — the composed outcome, not
the prose. Target: **4/4**, with the original three still passing.


In [ ]:
EXCHANGE_REQUEST = {
    "request": "A customer exchanges one blue mug for one tea pot.",
    "expect": lambda s, l: s["blue mug"]["stock"] == 11 and s["tea pot"]["stock"] == 1
              and any(r["action"] == "return" and r["product"] == "blue mug" for r in l)
              and any(r["action"] == "sale" and r["product"] == "tea pot" for r in l),
}

final_score = score(REQUESTS + [EXCHANGE_REQUEST], PLANNER_SYSTEM_V2,
                    extra_tools=make_exchange_tool)
print(f"\nfinal: {final_score}/4")


## 7. Completion Check

Submit: run the notebook top to bottom, then **File → Download → Download .ipynb**
and upload the file to the LMS.


In [ ]:
_store, _ledger = copy.deepcopy(STORE0), []
_doc = make_exchange_tool(_store, _ledger)["exchange_item"].__doc__ or ""
completion = {
    "exchange docstring written (>= 100 chars, has Args)":
        len(_doc.strip()) >= 100 and "Args" in _doc,
    "docstring states when NOT to use it":
        "not" in _doc.lower(),
    "planner extended with the exchange line": len(EXCHANGE_LINE.strip()) >= 30,
    "final score 4/4": final_score == 4,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nHOMEWORK COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")
